# PyTorch Custom Dataset and DataLoader: __getitem__, __len__, collate_fn, and num_workers

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/pytorch_3)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class TabularDataset(Dataset):
    """Simple dataset for tabular (features, label) data."""

    def __init__(self, features: np.ndarray, labels: np.ndarray):
        # Store as numpy arrays; convert to tensor in __getitem__
        assert len(features) == len(labels), "Feature/label count mismatch"
        self.features = features.astype(np.float32)
        self.labels = labels.astype(np.int64)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int) -> tuple:
        x = torch.from_numpy(self.features[idx])
        y = torch.tensor(self.labels[idx])
        return x, y

# Create dummy data
np.random.seed(42)
X = np.random.randn(1000, 20)
y = np.random.randint(0, 5, 1000)

dataset = TabularDataset(X, y)
print(f"Dataset length: {len(dataset)}")
print(f"Sample 0: features shape={dataset[0][0].shape}, label={dataset[0][1]}")
print(f"Sample dtype: {dataset[0][0].dtype}, {dataset[0][1].dtype}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.from_numpy(features.astype(np.float32))
        self.labels = torch.from_numpy(labels.astype(np.int64))
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

np.random.seed(42)
dataset = TabularDataset(np.random.randn(1000, 20), np.random.randint(0, 5, 1000))

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,   # single-process for now
    drop_last=False, # keep last incomplete batch
)

print(f"Total batches: {len(loader)}")

for batch_idx, (x_batch, y_batch) in enumerate(loader):
    print(f"Batch {batch_idx}: features={x_batch.shape}, labels={y_batch.shape}")
    if batch_idx == 2:
        break

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import time
import numpy as np

class SlowDataset(Dataset):
    """Simulates I/O-bound loading (e.g., reading from disk)."""
    def __init__(self, size=512):
        self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        # Simulate disk read latency
        time.sleep(0.001)
        return torch.randn(100), torch.randint(0, 10, ())

dataset = SlowDataset(512)

for nw in [0, 2, 4]:
    loader = DataLoader(dataset, batch_size=32, num_workers=nw)
    start = time.time()
    for _ in loader:
        pass
    elapsed = time.time() - start
    print(f"num_workers={nw}: {elapsed:.2f}s for {len(loader)} batches")

In [ ]:
# On Windows, always use this guard in your training script:
if __name__ == "__main__":
    loader = DataLoader(dataset, batch_size=32, num_workers=4)
    for batch in loader:
        pass  # safe

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np

class AugmentedDataset(Dataset):
    def __init__(self, size=100): self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        # Augmentation uses numpy random — problem without seeding workers
        noise = np.random.normal(0, 0.1, 10).astype(np.float32)
        return torch.from_numpy(noise), idx

def seed_worker(worker_id: int):
    """Give each worker a unique random seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    import random
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

loader = DataLoader(
    AugmentedDataset(),
    batch_size=4,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
)

batch1 = next(iter(loader))
batch2 = next(iter(loader))
print(f"Batches are different: {not torch.allclose(batch1[0], batch2[0])}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class VariableLengthDataset(Dataset):
    """Text sequences with different lengths."""
    def __init__(self):
        self.data = [
            (torch.tensor([1, 2, 3, 4, 5]), 0),
            (torch.tensor([10, 20]), 1),
            (torch.tensor([7, 8, 9, 4, 1, 2, 6]), 1),
            (torch.tensor([3, 1]), 0),
        ]
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

dataset = VariableLengthDataset()

# Default collate fails — sequences have different lengths
loader_default = DataLoader(dataset, batch_size=2)
try:
    next(iter(loader_default))
except RuntimeError as e:
    print(f"Default collate error: {str(e)[:80]}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class VariableLengthDataset(Dataset):
    def __init__(self):
        self.data = [
            (torch.tensor([1, 2, 3, 4, 5]), 0),
            (torch.tensor([10, 20]), 1),
            (torch.tensor([7, 8, 9, 4, 1, 2, 6]), 1),
            (torch.tensor([3, 1]), 0),
        ]
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_with_padding(batch: list) -> tuple:
    """
    Pad variable-length sequences to the max length in the batch.
    Returns padded sequences, lengths (for pack_padded_sequence), and labels.
    """
    sequences, labels = zip(*batch)

    # Sort by length (descending) — required by pack_padded_sequence
    lengths = torch.tensor([len(s) for s in sequences])
    sorted_idx = lengths.argsort(descending=True)
    sequences = [sequences[i] for i in sorted_idx]
    lengths = lengths[sorted_idx]
    labels = torch.tensor([labels[i] for i in sorted_idx])

    # Pad to longest sequence in batch (padding value = 0)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)

    return padded, lengths, labels

dataset = VariableLengthDataset()
loader = DataLoader(dataset, batch_size=3, collate_fn=collate_with_padding)

for padded, lengths, labels in loader:
    print(f"Padded shape:  {padded.shape}")
    print(f"Lengths:       {lengths}")
    print(f"Labels:        {labels}")
    print(f"Padded batch:\n{padded}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class ImageFolderDataset(Dataset):
    """
    Loads images from a directory structure:
    root/class_name/image.jpg
    """
    def __init__(self, root_dir: str, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}

        # Walk directory to find classes and images
        classes = sorted([d for d in os.listdir(root_dir)
                          if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_dir, fname), self.class_to_idx[cls]))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple:
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')  # always 3-channel
        if self.transform:
            image = self.transform(image)
        return image, label

# Transforms for training vs validation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Usage (requires actual image directories):
# train_ds = ImageFolderDataset("data/train", transform=train_transform)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
#                           num_workers=4, pin_memory=True, persistent_workers=True)
print("ImageFolderDataset class defined — ready to use with your data directory.")
print(f"Normalization: ImageNet mean/std (correct for pretrained models)")